# Credit Risk Scoring Model — Feature Engineering
### McMillin Analytics

**Objective:** Transform raw data into model-ready features using credit analysis domain knowledge.

The goal here is to create features that capture what an experienced underwriter looks for — not just feeding raw columns into a model, but engineering variables that reflect real credit risk logic.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Load from exploration notebook
df = pd.read_parquet('../data/processed/01_explored_data.parquet')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded: 1,345,350 rows x 74 columns


## 1. Feature Selection

Based on the exploration analysis, we'll select features organized by the 5 C's framework.
Only features available **at the time of underwriting** are included.

In [2]:
# Define feature groups based on credit analysis framework
capacity_features = ['annual_inc', 'dti', 'emp_length']

character_features = ['fico_range_high', 'fico_range_low', 'delinq_2yrs', 
                      'inq_last_6mths', 'pub_rec', 'pub_rec_bankruptcies',
                      'open_acc', 'total_acc', 'earliest_cr_line']

capital_features = ['home_ownership', 'revol_bal', 'revol_util', 
                    'total_rev_hi_lim', 'tot_cur_bal']

condition_features = ['loan_amnt', 'int_rate', 'term', 'grade', 'sub_grade',
                      'purpose', 'installment']

# Additional features that may add value
other_features = ['addr_state', 'application_type', 'mort_acc',
                  'num_actv_bc_tl', 'num_bc_tl', 'num_sats',
                  'pct_tl_nvr_dlq', 'tot_hi_cred_lim',
                  'total_bal_ex_mort', 'total_bc_limit',
                  'total_il_high_credit_limit']

# Combine and filter to columns that actually exist
all_candidate_features = (capacity_features + character_features + 
                          capital_features + condition_features + other_features)
selected_features = [f for f in all_candidate_features if f in df.columns]
missing_features = [f for f in all_candidate_features if f not in df.columns]

print(f'Selected features: {len(selected_features)}')
if missing_features:
    print(f'Not available in dataset: {missing_features}')

# Create working dataframe
df_model = df[selected_features + ['default']].copy()

Selected features: 35


## 2. Derived Features

These are engineered features that reflect actual underwriting logic — the kind of calculations a credit analyst does mentally when reviewing an application.

In [3]:
# --- CAPACITY-BASED FEATURES ---

# Payment-to-Income Ratio
# How much of monthly income goes to this specific loan payment
# In commercial lending, we look at DSCR; this is the consumer equivalent
df_model['payment_to_income'] = (
    df_model['installment'] / (df_model['annual_inc'] / 12)
) * 100

# Loan-to-Income Ratio  
# Total loan exposure relative to annual earnings
df_model['loan_to_income'] = df_model['loan_amnt'] / df_model['annual_inc']

print('Capacity features created:')
print(f'  payment_to_income — median: {df_model["payment_to_income"].median():.1f}%')
print(f'  loan_to_income    — median: {df_model["loan_to_income"].median():.2f}x')

Capacity features created:
  payment_to_income — median: 7.2%
  loan_to_income    — median: 0.20x


In [4]:
# Pro Forma DTI — total debt burden INCLUDING proposed loan payment
# This mirrors what an underwriter calculates: "what does the borrower's
# debt load look like after taking on this new obligation?"
existing_monthly_debt = (df_model['dti'] / 100) * (df_model['annual_inc'] / 12)
total_monthly_debt = existing_monthly_debt + df_model['installment']
df_model['dti_pro_forma'] = (total_monthly_debt / (df_model['annual_inc'] / 12)) * 100

print(f'  dti_pro_forma     — median: {df_model["dti_pro_forma"].median():.1f}%')

  dti_pro_forma     — median: 25.4%


In [5]:
# --- CHARACTER-BASED FEATURES ---

# FICO midpoint (cleaner single feature)
df_model['fico_score'] = (df_model['fico_range_low'] + df_model['fico_range_high']) / 2

# Credit history length in years
if 'earliest_cr_line' in df_model.columns:
    df_model['earliest_cr_line'] = pd.to_datetime(df_model['earliest_cr_line'], errors='coerce')
    # Use a reference date (approximate dataset midpoint)
    ref_date = pd.Timestamp('2015-01-01')
    df_model['credit_history_years'] = (
        (ref_date - df_model['earliest_cr_line']).dt.days / 365.25
    )
    df_model = df_model.drop(columns=['earliest_cr_line'])

# Total derogatory marks (composite risk flag)
derog_cols = ['delinq_2yrs', 'pub_rec', 'pub_rec_bankruptcies']
derog_available = [c for c in derog_cols if c in df_model.columns]
df_model['total_derog_marks'] = df_model[derog_available].fillna(0).sum(axis=1)

# Inquiry intensity — many recent inquiries signal a borrower shopping for credit (stress indicator)
if 'inq_last_6mths' in df_model.columns:
    df_model['high_inquiry_flag'] = (df_model['inq_last_6mths'] >= 3).astype(int)

# Account utilization rate (open accounts / total accounts)
if 'open_acc' in df_model.columns and 'total_acc' in df_model.columns:
    df_model['account_util_rate'] = df_model['open_acc'] / df_model['total_acc'].replace(0, np.nan)

print('Character features created.')

Character features created.


In [6]:
# --- CAPITAL / LEVERAGE FEATURES ---

# Revolving utilization (ensure numeric)
if df_model['revol_util'].dtype == 'object':
    df_model['revol_util'] = df_model['revol_util'].str.rstrip('%').astype(float)

# Revolving balance to income
df_model['revol_bal_to_income'] = df_model['revol_bal'] / df_model['annual_inc'].replace(0, np.nan)

# Available revolving credit (headroom)
if 'total_rev_hi_lim' in df_model.columns:
    df_model['revol_headroom'] = df_model['total_rev_hi_lim'] - df_model['revol_bal']

# Total leverage: all balances relative to income
if 'tot_cur_bal' in df_model.columns:
    df_model['total_leverage'] = df_model['tot_cur_bal'] / df_model['annual_inc'].replace(0, np.nan)

print('Capital/leverage features created.')

Capital/leverage features created.


In [7]:
# --- CONDITION-BASED FEATURES ---

# Rate premium over grade average — is this loan priced high even for its grade?
if df_model['int_rate'].dtype == 'object':
    df_model['int_rate'] = df_model['int_rate'].str.rstrip('%').astype(float)

grade_avg_rate = df_model.groupby('grade')['int_rate'].transform('mean')
df_model['rate_premium'] = df_model['int_rate'] - grade_avg_rate

# Loan amount relative to income (affordability check)
# Already created as loan_to_income above

print('Condition features created.')
print(f'\nTotal features after engineering: {df_model.shape[1] - 1}')  # -1 for target

Condition features created.

Total features after engineering: 46


## 3. Encode Categorical Variables

In [8]:
# Identify categorical columns
cat_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical features to encode: {cat_cols}')

for col in cat_cols:
    print(f'\n{col}: {df_model[col].nunique()} unique values')
    print(f'  Top values: {df_model[col].value_counts().head(5).to_dict()}')

Categorical features to encode: ['emp_length', 'home_ownership', 'term', 'grade', 'sub_grade', 'purpose', 'addr_state', 'application_type']

emp_length: 11 unique values
  Top values: {'10+ years': 442209, '2 years': 121751, '< 1 year': 108065, '3 years': 107602, '1 year': 88495}

home_ownership: 6 unique values
  Top values: {'MORTGAGE': 665596, 'RENT': 534436, 'OWN': 144840, 'ANY': 286, 'OTHER': 144}

term: 2 unique values
  Top values: {' 36 months': 1020768, ' 60 months': 324582}

grade: 7 unique values
  Top values: {'B': 392748, 'C': 381694, 'A': 235095, 'D': 200966, 'E': 93656}

sub_grade: 35 unique values
  Top values: {'C1': 85496, 'B4': 83200, 'B5': 82541, 'B3': 81828, 'C2': 79215}

purpose: 14 unique values
  Top values: {'debt_consolidation': 780342, 'credit_card': 295285, 'home_improvement': 87507, 'other': 77877, 'major_purchase': 29427}

addr_state: 51 unique values
  Top values: {'CA': 196529, 'TX': 110173, 'NY': 109849, 'FL': 95611, 'IL': 51723}

application_type: 2 un

In [9]:
# Employment length — convert to numeric
if 'emp_length' in df_model.columns:
    emp_map = {
        '< 1 year': 0.5, '1 year': 1, '2 years': 2, '3 years': 3,
        '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
        '8 years': 8, '9 years': 9, '10+ years': 10
    }
    df_model['emp_length_num'] = df_model['emp_length'].map(emp_map)
    df_model = df_model.drop(columns=['emp_length'])

# Term — extract numeric
if 'term' in df_model.columns:
    df_model['term_months'] = df_model['term'].str.extract('(\d+)').astype(float)
    df_model = df_model.drop(columns=['term'])

# Grade — ordinal encode (A=1, B=2, ... G=7)
if 'grade' in df_model.columns:
    grade_map = {g: i+1 for i, g in enumerate('ABCDEFG')}
    df_model['grade_num'] = df_model['grade'].map(grade_map)
    df_model = df_model.drop(columns=['grade'])

# Sub-grade — ordinal encode (A1=1, A2=2, ... G5=35)
if 'sub_grade' in df_model.columns:
    sub_grades = sorted(df_model['sub_grade'].dropna().unique())
    sub_grade_map = {sg: i+1 for i, sg in enumerate(sub_grades)}
    df_model['sub_grade_num'] = df_model['sub_grade'].map(sub_grade_map)
    df_model = df_model.drop(columns=['sub_grade'])

print('Ordinal features encoded.')

Ordinal features encoded.


In [10]:
# Home ownership — one-hot encode (few categories)
if 'home_ownership' in df_model.columns:
    # Consolidate rare categories
    df_model['home_ownership'] = df_model['home_ownership'].replace(
        {'ANY': 'OTHER', 'NONE': 'OTHER'}
    )
    home_dummies = pd.get_dummies(df_model['home_ownership'], prefix='home', drop_first=True)
    df_model = pd.concat([df_model, home_dummies], axis=1)
    df_model = df_model.drop(columns=['home_ownership'])

# Purpose — one-hot encode (group rare purposes)
if 'purpose' in df_model.columns:
    # Keep top purposes, group rest as 'other'
    top_purposes = df_model['purpose'].value_counts().head(8).index
    df_model['purpose_clean'] = df_model['purpose'].where(
        df_model['purpose'].isin(top_purposes), 'other'
    )
    purpose_dummies = pd.get_dummies(df_model['purpose_clean'], prefix='purpose', drop_first=True)
    df_model = pd.concat([df_model, purpose_dummies], axis=1)
    df_model = df_model.drop(columns=['purpose', 'purpose_clean'])

# Application type
if 'application_type' in df_model.columns:
    df_model['is_joint'] = (df_model['application_type'] == 'Joint App').astype(int)
    df_model = df_model.drop(columns=['application_type'])

# State — too many categories for one-hot; use target encoding or drop
# For now, drop it (can revisit with target encoding later)
if 'addr_state' in df_model.columns:
    df_model = df_model.drop(columns=['addr_state'])

print('Categorical encoding complete.')

Categorical encoding complete.


In [11]:
# Drop original FICO range columns (we have fico_score)
for col in ['fico_range_low', 'fico_range_high']:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# Drop any remaining non-numeric columns
remaining_obj = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
if remaining_obj:
    print(f'Dropping remaining non-numeric: {remaining_obj}')
    df_model = df_model.drop(columns=remaining_obj)

print(f'\nFinal feature set: {df_model.shape[1] - 1} features')
print(f'Samples: {df_model.shape[0]:,}')


Final feature set: 51 features
Samples: 1,345,350


## 4. Handle Missing Values

In [12]:
# Assess remaining missing values
missing = df_model.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print('Remaining missing values:')
for col, count in missing.items():
    print(f'  {col:.<40} {count:>8,} ({count/len(df_model)*100:.1f}%)')

Remaining missing values:
  emp_length_num..........................   78,516 (5.8%)
  total_leverage..........................   67,888 (5.0%)
  pct_tl_nvr_dlq..........................   67,681 (5.0%)
  tot_hi_cred_lim.........................   67,527 (5.0%)
  total_il_high_credit_limit..............   67,527 (5.0%)
  tot_cur_bal.............................   67,527 (5.0%)
  num_actv_bc_tl..........................   67,527 (5.0%)
  num_bc_tl...............................   67,527 (5.0%)
  revol_headroom..........................   67,527 (5.0%)
  total_rev_hi_lim........................   67,527 (5.0%)
  num_sats................................   55,841 (4.2%)
  mort_acc................................   47,281 (3.5%)
  total_bal_ex_mort.......................   47,281 (3.5%)
  total_bc_limit..........................   47,281 (3.5%)
  revol_util..............................      857 (0.1%)
  pub_rec_bankruptcies....................      697 (0.1%)
  dti_pro_forma...............

In [13]:
# Strategy: 
# - Numeric features with <5% missing: fill with median
# - Features with >20% missing: consider dropping
# - Employment length missing: fill with median (likely "not reported")

for col in df_model.columns:
    if df_model[col].isnull().sum() > 0:
        missing_pct = df_model[col].isnull().mean()
        if missing_pct > 0.20:
            print(f'Dropping {col} ({missing_pct*100:.1f}% missing)')
            df_model = df_model.drop(columns=[col])
        else:
            median_val = df_model[col].median()
            df_model[col] = df_model[col].fillna(median_val)

# Verify no missing values remain
assert df_model.isnull().sum().sum() == 0, 'Missing values remain!'
print(f'\nAll missing values handled. Final shape: {df_model.shape}')


All missing values handled. Final shape: (1345350, 52)


## 5. Handle Infinite Values & Outliers

In [14]:
# Replace infinities
df_model = df_model.replace([np.inf, -np.inf], np.nan)
for col in df_model.columns:
    if df_model[col].isnull().sum() > 0:
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Cap extreme outliers at 1st and 99th percentile for key features
# (prevents a few extreme values from dominating the model)
cap_features = ['annual_inc', 'revol_bal', 'loan_to_income', 
                'payment_to_income', 'revol_bal_to_income', 'total_leverage']
cap_features = [f for f in cap_features if f in df_model.columns]

for col in cap_features:
    lower = df_model[col].quantile(0.01)
    upper = df_model[col].quantile(0.99)
    df_model[col] = df_model[col].clip(lower, upper)

print('Outliers capped at 1st/99th percentile for key features.')

Outliers capped at 1st/99th percentile for key features.


## 6. Train/Test Split

In [15]:
# Separate features and target
X = df_model.drop(columns=['default'])
y = df_model['default']

# 80/20 split with stratification (maintain default rate in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]:,} samples ({y_train.mean()*100:.1f}% default rate)')
print(f'Test set:      {X_test.shape[0]:,} samples ({y_test.mean()*100:.1f}% default rate)')
print(f'Features:      {X_train.shape[1]}')

# Save for modeling notebook
X_train.to_parquet('../data/processed/X_train.parquet', index=False)
X_test.to_parquet('../data/processed/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/processed/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/processed/y_test.parquet', index=False)

# Also save feature names for the Streamlit app
pd.Series(X_train.columns).to_csv('../data/processed/feature_names.csv', index=False)

print('\nData saved for modeling notebook.')

Training set:  1,076,280 samples (20.0% default rate)
Test set:      269,070 samples (20.0% default rate)
Features:      51

Data saved for modeling notebook.


In [16]:
# Final feature list summary
print('=' * 60)
print('FINAL FEATURE SET')
print('=' * 60)
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i:>2}. {col}')

FINAL FEATURE SET
   1. annual_inc
   2. dti
   3. delinq_2yrs
   4. inq_last_6mths
   5. pub_rec
   6. pub_rec_bankruptcies
   7. open_acc
   8. total_acc
   9. revol_bal
  10. revol_util
  11. total_rev_hi_lim
  12. tot_cur_bal
  13. loan_amnt
  14. int_rate
  15. installment
  16. mort_acc
  17. num_actv_bc_tl
  18. num_bc_tl
  19. num_sats
  20. pct_tl_nvr_dlq
  21. tot_hi_cred_lim
  22. total_bal_ex_mort
  23. total_bc_limit
  24. total_il_high_credit_limit
  25. payment_to_income
  26. loan_to_income
  27. dti_pro_forma
  28. fico_score
  29. credit_history_years
  30. total_derog_marks
  31. high_inquiry_flag
  32. account_util_rate
  33. revol_bal_to_income
  34. revol_headroom
  35. total_leverage
  36. rate_premium
  37. emp_length_num
  38. term_months
  39. grade_num
  40. sub_grade_num
  41. home_OTHER
  42. home_OWN
  43. home_RENT
  44. purpose_credit_card
  45. purpose_debt_consolidation
  46. purpose_home_improvement
  47. purpose_major_purchase
  48. purpose_medical
 